# EDA — eksploracyjna analiza danych

Projekt dotyczy analizy komentarzy użytkowników Reddita w wybranych społecznościach muzycznych. Celem notebooka jest uporządkowanie danych wejściowych, opisanie podstawowych cech zbioru oraz wskazanie różnic między subredditami przed przejściem do analizy text mining i analizy sieci społecznościowych.


## 1. Opis zbioru danych

Dane wykorzystane w projekcie pochodzą z Reddit API i obejmują komentarze z czterech społeczności muzycznych: `r/hiphopheads`, `r/indieheads`, `r/Metal` oraz `r/popheads`. Zbiór obejmuje okres od 11 marca 2020 r. do 11 marca 2021 r.

Analiza EDA ma na celu:

- określenie wielkości i struktury zbioru danych,
- sprawdzenie liczby komentarzy i aktywności użytkowników w poszczególnych subredditach,
- porównanie długości komentarzy oraz poziomu zaangażowania,
- przygotowanie oczyszczonego zbioru do dalszej analizy tekstowej i sieciowej.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Ustawienia wizualizacji
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

# Ścieżki projektu — notebook zakłada typową strukturę: /notebooks, /data, /visualizations, /reports
DATA_PATH_CANDIDATES = [
    Path("../data/processed/all_subreddits_sample.csv"),
    Path("data/processed/all_subreddits_sample.csv"),
    Path("all_subreddits_sample.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])
VIS_DIR = Path("../visualizations")
REPORTS_DIR = Path("../reports")
VIS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wczytywany plik: {DATA_PATH}")


In [ ]:
df_raw = pd.read_csv(DATA_PATH)

# Ujednolicenie nazw kolumn, jeśli w danych występują warianty nazw
if "id" in df_raw.columns and "comment_id" not in df_raw.columns:
    df_raw = df_raw.rename(columns={"id": "comment_id"})

required_columns = {"subreddit", "author", "body", "created_utc"}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"Brakuje wymaganych kolumn: {missing_columns}")

print(f"Liczba rekordów przed czyszczeniem: {len(df_raw):,}")
display(df_raw.head())


## 2. Czyszczenie danych

Na początku analizy usunięto rekordy, które mogłyby zaburzać wyniki: komentarze usunięte przez użytkowników lub moderatorów oraz konta botów. Dzięki temu wszystkie dalsze statystyki odnoszą się do tego samego, oczyszczonego zbioru danych.


In [ ]:
df = df_raw.copy()

# Usuwanie pustych i technicznych treści komentarzy
body_clean = df["body"].astype(str).str.strip()
removed_comment_mask = body_clean.str.lower().isin(["[deleted]", "[removed]", "deleted", "removed", ""])

# Usuwanie autorów technicznych i botów
known_bots = {
    "AutoModerator",
    "autotldr",
    "tweettranscriberbot",
    "transcribersofreddit",
    "RemindMeBot",
}
author_clean = df["author"].astype(str).str.strip()
bot_mask = (
    author_clean.isin(known_bots)
    | author_clean.str.lower().str.contains("bot", na=False)
    | author_clean.str.lower().isin(["[deleted]", "deleted", "automoderator"])
)

cleaning_summary = pd.DataFrame({
    "kategoria": ["rekordy początkowe", "komentarze usunięte", "boty/konta techniczne", "rekordy po czyszczeniu"],
    "liczba": [len(df), int(removed_comment_mask.sum()), int(bot_mask.sum()), int((~removed_comment_mask & ~bot_mask).sum())]
})

df = df.loc[~removed_comment_mask & ~bot_mask].copy()

# Daty i zmienne pomocnicze
df["date"] = pd.to_datetime(pd.to_numeric(df["created_utc"], errors="coerce"), unit="s", errors="coerce")
df = df.dropna(subset=["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.to_period("M")
df["body_len"] = df["body"].astype(str).str.len()
df["word_count"] = df["body"].astype(str).str.split().str.len()

cleaning_summary.to_csv(REPORTS_DIR / "eda_cleaning_summary.csv", index=False)
display(cleaning_summary)
print(f"Liczba rekordów po czyszczeniu i usunięciu błędnych dat: {len(df):,}")


Komentarze techniczne i automatyczne mogą sztucznie zwiększać aktywność wybranych społeczności, dlatego zostały usunięte przed obliczeniem statystyk. Taki zabieg poprawia porównywalność subredditów i zmniejsza ryzyko, że wyniki będą opisywać pracę botów, a nie zachowania użytkowników.


## 3. Struktura zbioru według subredditów

Poniższa tabela pokazuje podstawowy rozkład komentarzy między społecznościami. Oprócz liczby komentarzy uwzględniono udział procentowy danego subreddita w zbiorze oraz średnią długość komentarza.


In [ ]:
subreddit_summary = (
    df.groupby("subreddit")
    .agg(
        liczba_komentarzy=("body", "count"),
        unikalni_autorzy=("author", "nunique"),
        srednia_dlugosc_komentarza=("body_len", "mean"),
        mediana_dlugosci_komentarza=("body_len", "median"),
        srednia_liczba_slow=("word_count", "mean"),
    )
    .sort_values("liczba_komentarzy", ascending=False)
)
subreddit_summary["udzial_%"] = (subreddit_summary["liczba_komentarzy"] / subreddit_summary["liczba_komentarzy"].sum() * 100)
subreddit_summary = subreddit_summary[
    ["liczba_komentarzy", "udzial_%", "unikalni_autorzy", "srednia_dlugosc_komentarza", "mediana_dlugosci_komentarza", "srednia_liczba_slow"]
].round(2)

subreddit_summary.to_csv(REPORTS_DIR / "eda_subreddit_summary.csv")
display(subreddit_summary)


In [ ]:
fig, ax = plt.subplots()
plot_data = subreddit_summary.reset_index().sort_values("liczba_komentarzy", ascending=True)
ax.barh(plot_data["subreddit"], plot_data["liczba_komentarzy"])
ax.set_title("Liczba komentarzy według subreddita")
ax.set_xlabel("Liczba komentarzy")
ax.set_ylabel("Subreddit")
plt.tight_layout()
plt.savefig(VIS_DIR / "eda_comments_by_subreddit.png", dpi=150, bbox_inches="tight")
plt.show()


Tabela i wykres pozwalają ocenić, czy zbiór jest zrównoważony. Jeżeli jedna społeczność ma wyraźnie większy udział w danych, należy uwzględnić to przy interpretacji późniejszych wyników text mining i analizy sieciowej, ponieważ większa liczba komentarzy może zwiększać widoczność danego subreddita w wynikach zbiorczych.


## 4. Aktywność w czasie

Analiza aktywności miesięcznej pozwala sprawdzić, czy liczba komentarzy była stabilna, czy też występowały okresowe wzrosty aktywności. Takie zmiany mogą wynikać np. z premier albumów, wydarzeń branżowych, kontrowersji albo sezonowego wzrostu dyskusji.


In [ ]:
activity = (
    df.groupby(["subreddit", "month"])
    .size()
    .reset_index(name="liczba_komentarzy")
)
activity["month_ts"] = activity["month"].dt.to_timestamp()
activity.to_csv(REPORTS_DIR / "eda_monthly_activity.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
for sub, group in activity.groupby("subreddit"):
    group = group.sort_values("month_ts")
    ax.plot(group["month_ts"], group["liczba_komentarzy"], marker="o", linewidth=2, label=f"r/{sub}")

ax.set_title("Aktywność miesięczna w analizowanych subredditach")
ax.set_xlabel("Miesiąc")
ax.set_ylabel("Liczba komentarzy")
ax.legend(title="Subreddit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(VIS_DIR / "eda_monthly_activity.png", dpi=150, bbox_inches="tight")
plt.show()


Wykres aktywności czasowej pokazuje, czy dyskusje w poszczególnych społecznościach rozwijały się równomiernie. Wyraźne skoki aktywności warto interpretować ostrożnie, ponieważ mogą być związane z pojedynczymi wydarzeniami, które wpływają na całościowy obraz społeczności.


## 5. Długość komentarzy

Długość komentarzy jest prostym wskaźnikiem charakteru dyskusji. Krótsze komentarze mogą sugerować bardziej reaktywny styl komunikacji, natomiast dłuższe wypowiedzi mogą wskazywać na bardziej opisowe, argumentacyjne lub recenzenckie rozmowy.


In [ ]:
length_stats = (
    df.groupby("subreddit")["body_len"]
    .agg(srednia="mean", mediana="median", min="min", max="max", q95=lambda x: x.quantile(0.95))
    .round(1)
)
length_stats.to_csv(REPORTS_DIR / "eda_comment_length_stats.csv")
display(length_stats)


In [ ]:
# Przycięcie do 99 percentyla poprawia czytelność wykresu, nie zmieniając głównej interpretacji rozkładu.
cutoff = df["body_len"].quantile(0.99)
plot_df = df.loc[df["body_len"] <= cutoff].copy()

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(
    data=plot_df,
    x="body_len",
    hue="subreddit",
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.set_title("Rozkład długości komentarzy według subredditów")
ax.set_xlabel("Długość komentarza — liczba znaków")
ax.set_ylabel("Gęstość")
plt.tight_layout()
plt.savefig(VIS_DIR / "eda_comment_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


Porównanie długości komentarzy pomaga określić, czy społeczności różnią się stylem dyskusji. Różnice w medianie są zwykle bardziej odporne na pojedyncze bardzo długie komentarze niż średnia, dlatego oba wskaźniki powinny być analizowane łącznie.


## 6. Wynik komentarzy i poziom zaangażowania

Wynik komentarza (`score`) może być traktowany jako przybliżony wskaźnik reakcji społeczności. Nie jest to jednak bezpośrednia miara jakości komentarza, ponieważ zależy od widoczności wpisu, czasu publikacji, popularności wątku oraz norm danej społeczności.


In [ ]:
if "score" in df.columns:
    score_stats = (
        df.groupby("subreddit")["score"]
        .agg(srednia="mean", mediana="median", min="min", max="max", q95=lambda x: x.quantile(0.95))
        .round(2)
    )
    score_stats.to_csv(REPORTS_DIR / "eda_score_stats.csv")
    display(score_stats)
else:
    display(Markdown("W zbiorze danych nie ma kolumny `score`, dlatego pominięto analizę wyników komentarzy."))


In [ ]:
if "score" in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for i, (sub, group) in enumerate(df.groupby("subreddit")):
        low = group["score"].quantile(0.01)
        high = group["score"].quantile(0.99)
        data = group.loc[(group["score"] >= low) & (group["score"] <= high), "score"]
        axes[i].hist(data, bins=50)
        axes[i].set_title(f"r/{sub}")
        axes[i].set_xlabel("Score")
        axes[i].set_ylabel("Liczba komentarzy")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Rozkład wyników komentarzy według subredditów", fontsize=16)
    plt.tight_layout()
    plt.savefig(VIS_DIR / "eda_score_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()


Rozkład wyników pozwala sprawdzić, czy w danych dominują komentarze neutralnie oceniane, czy też pojawia się większa liczba komentarzy szczególnie wysoko ocenionych. Przy interpretacji należy pamiętać, że score jest zależny od algorytmów Reddita oraz czasu ekspozycji komentarza.


## 7. Najaktywniejsi autorzy

Analiza najaktywniejszych autorów pozwala ocenić, czy dyskusje są rozproszone między wielu użytkowników, czy zdominowane przez niewielką grupę bardzo aktywnych kont.


In [ ]:
top_authors_tables = []

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (sub, group) in enumerate(df.groupby("subreddit")):
    top = group["author"].value_counts().head(10)
    top_table = top.rename_axis("author").reset_index(name="liczba_komentarzy")
    top_table.insert(0, "subreddit", sub)
    top_authors_tables.append(top_table)

    axes[i].barh(top.index[::-1], top.values[::-1])
    axes[i].set_title(f"r/{sub} — top 10 autorów")
    axes[i].set_xlabel("Liczba komentarzy")
    axes[i].set_ylabel("Autor")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(VIS_DIR / "eda_top_authors.png", dpi=150, bbox_inches="tight")
plt.show()

top_authors_df = pd.concat(top_authors_tables, ignore_index=True)
top_authors_df.to_csv(REPORTS_DIR / "eda_top_authors.csv", index=False)
display(top_authors_df.head(20))


Jeżeli w danym subreddicie kilku autorów generuje dużą część komentarzy, może to wpływać na wyniki późniejszej analizy tekstowej i sieciowej. W takim przypadku część obserwowanych wzorców może odzwierciedlać styl komunikacji najbardziej aktywnych użytkowników, a nie całej społeczności.


## 8. Kontrowersyjność i wyróżnienia

Kolumny `controversiality` i `gilded`, jeżeli występują w zbiorze, pozwalają dodatkowo opisać charakter reakcji społeczności. Kontrowersyjność wskazuje komentarze, które mogły wywoływać podzielone oceny, a `gilded` informuje o wyróżnieniach przyznanych przez użytkowników.


In [ ]:
additional_metrics = []

if "controversiality" in df.columns:
    controversial = (
        df.groupby("subreddit")["controversiality"]
        .mean()
        .mul(100)
        .round(2)
        .rename("odsetek_kontrowersyjnych_%")
    )
    additional_metrics.append(controversial)

if "gilded" in df.columns:
    gilded = df.groupby("subreddit")["gilded"].sum().rename("gilded_lacznie")
    additional_metrics.append(gilded)

if additional_metrics:
    additional_metrics_df = pd.concat(additional_metrics, axis=1)
    additional_metrics_df.to_csv(REPORTS_DIR / "eda_additional_metrics.csv")
    display(additional_metrics_df)
else:
    display(Markdown("W zbiorze nie ma kolumn `controversiality` ani `gilded`, dlatego pominięto tę część analizy."))


Dodatkowe metryki mogą wskazywać na różnice w sposobie reagowania społeczności na komentarze. Należy jednak traktować je jako uzupełnienie, ponieważ pojedynczy wskaźnik nie wyjaśnia pełnego kontekstu dyskusji.


## 9. Wnioski z analizy eksploracyjnej

Analiza eksploracyjna pokazuje podstawową strukturę danych i pozwala przygotować zbiór do dalszych etapów projektu. Najważniejsze wnioski należy interpretować w odniesieniu do liczebności subredditów, aktywności użytkowników i długości komentarzy.

Z perspektywy dalszej analizy szczególnie istotne są trzy kwestie. Po pierwsze, ewentualne różnice w liczbie komentarzy między subredditami mogą wpływać na wyniki zbiorcze, dlatego w analizie text mining warto porównywać społeczności osobno. Po drugie, długość komentarzy i aktywność najczęstszych autorów mogą wskazywać na odmienne style komunikacji w fandomach muzycznych. Po trzecie, aktywność czasowa może ujawniać okresowe wzrosty dyskusji, które warto uwzględniać przy interpretacji tematów i sentymentu.

Wygenerowane tabele zostały zapisane w folderze `reports`, a wykresy w folderze `visualizations`.
